# FX Hedging as a Contagion Channel:
# How NBFI Currency Risk Management Transmits Global Financial Shocks

## Project 4 — Standalone Research Module

---

**Abstract**

We investigate a novel contagion channel operating through the FX hedging activity of
non-bank financial intermediaries (NBFIs). Combining insights from Rey, Stavrakeva &
Tang (2024) — who show that equity market shocks drive exchange rate movements through
a "currency centrality" network — with Nenova, Schrimpf & Shin (2025) — who demonstrate
that FX swap volumes proxy for NBFI hedging of cross-border bond investments — we identify
a transmission chain: **equity shocks → FX movements → hedging cost changes → NBFI portfolio
adjustment → cross-border bond flow reversal → feedback to asset prices**.

Critically, we show that this channel exhibits **tail asymmetry**: in normal times, yield
curve movements partially offset FX-induced hedging cost changes (self-stabilizing); in
stress episodes, all forces reinforce — USD appreciation, curve flattening, and CIP
deviations blow out simultaneously — creating amplification. We detect this nonlinearity
using quantile methods (quantile regression, Delta-CoVaR, and quantile connectedness).

**Key references:**
- Rey, H., Stavrakeva, V. & Tang, J. (2024). *Currency centrality in equity markets,
  exchange rates and global financial cycles.* NBER WP 33003.
- Nenova, T., Schrimpf, A. & Shin, H.S. (2025). *Global portfolio investments and FX
  derivatives.* BIS Working Paper No. 1273.
- Ando, T., Greenwood-Nimmo, M. & Shin, Y. (2022). *Quantile connectedness.* Management
  Science, 68(4), 2401–2431.

## 1. Introduction & Motivation

The global FX derivatives market has grown to **$75 trillion** in outstanding notional
(BIS OTC statistics, end-2024), driven primarily by non-bank financial intermediaries —
investment funds, pension funds, and insurance companies — hedging cross-border bond
investments. This growth has created a powerful but underappreciated channel for the
international transmission of financial shocks.

### The Economic Mechanism

Two recent papers illuminate complementary pieces of this mechanism:

1. **Rey, Stavrakeva & Tang (2024)** decompose exchange rates into "equity net currency
   supplies" — local stock market capitalization minus foreign equity holdings. Their
   framework explains 95% of monthly FX variation vs the USD, showing that **equity
   shocks drive exchange rates** (not the reverse). The USD plays a uniquely central
   role in transmitting risk aversion globally.

2. **Nenova, Schrimpf & Shin (2025)** show that FX swap volumes are a **barometer of
   NBFI risk-taking**. They develop a portfolio choice model where yield curve slopes
   determine hedging demand: a steeper US curve → more hedged foreign investment in
   US bonds → more FX swap activity. Crucially, NBFIs use **short-term FX swaps
   (3 months) to hedge long-term bond positions (10 years)** — creating rollover risk
   and maturity mismatch vulnerability.

### The Offsetting Forces Hypothesis

A key insight is that equity shocks and yield curve movements create **opposing forces**
on FX hedging costs in normal times:

| Force | Normal Times | Stress |
|-------|-------------|--------|
| **Equity shock → FX** (Rey et al.) | USD appreciates moderately → hedging costs rise | USD appreciates violently → hedging costs spike |
| **Yield curve** (Nenova et al.) | Curve steepens → attractive hedged carry | Curve flattens/inverts → carry disappears |
| **CIP deviations** | Small → manageable cost | Blow out → prohibitive cost |
| **Net effect on hedging** | Partially offsetting → self-stabilizing | All reinforcing → amplification |

This asymmetry — offsetting at the median, reinforcing in the tails — is the core
empirical prediction that we test using quantile methods.

## 2. Conceptual Framework: The FX Hedging Amplification Channel

### Transmission Chain

```
1. Shock hits equity markets (e.g., US rate hike, risk-off event)
       ↓
2. Exchange rates move (Rey et al. — currency centrality mechanism)
       ↓
3. FX hedging costs spike for NBFIs (Nenova et al. — CIP deviations widen)
       ↓
4. NBFIs face margin calls on FX derivatives / rollover risk
       ↓
5. NBFIs deleverage cross-border bond positions to reduce hedge exposure
       ↓
6. Bond sell-off in destination country → yield spikes → further FX moves
       ↓
7. Feedback loop: steps 2–6 repeat with dampening
```

### Three Empirical Layers

| Layer | Method | What It Tests |
|-------|--------|---------------|
| **Layer 1** | Quantile regression of CIP basis | Offsetting forces vanish in tails |
| **Layer 2** | Delta-CoVaR: bond flows \| FX stress | Systemic risk contribution of FX hedging |
| **Layer 3** | Quantile connectedness network | Transmission chain activates in stress |

In [ ]:
# ============================================================================
# Setup and Imports
# ============================================================================
import sys
sys.path.insert(0, '/home/user/SergioSola')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.api as sm
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.analysis.fx_hedging_contagion import (
    build_synthetic_fx_hedging_panel,
    compute_hedging_cost_decomposition,
    quantile_cip_regression,
    asymmetry_test,
    covar_fx_hedging,
    delta_covar_all_currencies,
    build_connectedness_system,
    fx_hedging_quantile_connectedness,
    tail_connectedness_comparison,
    cross_currency_connectedness,
    nbfi_fx_hedging_panel_regression,
    iv_fx_hedging_regression,
    simulate_fx_hedging_cascade,
)

# Style
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'figure.dpi': 100,
})
COLORS = ['#1a2850', '#2864a0', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

print('All modules loaded successfully.')

## 3. Data Construction

We generate synthetic data that mirrors the structure and properties of real-world
data sources. The DGP embeds the **key economic mechanism**: offsetting forces at the
median, reinforcing forces in the tails.

### Data sources (synthetic equivalents):

| Dataset | Real Source | Frequency | Period |
|---------|-----------|-----------|--------|
| FX swap basis / CIP deviations | Bloomberg, Refinitiv | Monthly | 2005–2024 |
| Yield curve slopes (10Y - 2Y) | FRED, ECB SDW, BoJ | Monthly | 2005–2024 |
| Equity returns | MSCI, S&P 500 | Monthly | 2005–2024 |
| Cross-border bond flows | TIC (US Treasury), BIS IBS | Monthly | 2005–2024 |
| FX swap outstanding notional | BIS OTC derivatives | Semi-annual | 1998–2024 |
| NBFI assets (% of GDP) | FSB NBFI Monitor | Annual | 2002–2024 |
| VIX | CBOE | Monthly | 2005–2024 |
| Monetary policy shocks | High-frequency (Gürkaynak et al.) | Event | 2005–2024 |

In [ ]:
# ============================================================================
# 3.1 Generate Synthetic Panel
# ============================================================================
data = build_synthetic_fx_hedging_panel(n_periods=240, seed=42)
ts_data = data['ts_data']
country_panel = data['country_panel']
currencies = data['currencies']

print(f'Time series panel: {ts_data.shape[0]:,} observations')
print(f'  Currencies: {currencies}')
print(f'  Date range: {ts_data["date"].min().strftime("%Y-%m")} to '
      f'{ts_data["date"].max().strftime("%Y-%m")}')
print(f'  Variables: {list(ts_data.columns)}')
print(f'\nCountry panel: {country_panel.shape[0]:,} observations')
print(f'\nKey statistics:')
ts_data[['us_equity_return', 'fx_return', 'cip_basis', 'bond_flows',
         'vix', 'us_yield_slope']].describe().round(4)

In [ ]:
# ============================================================================
# 3.2 Visualize Key Time Series (EUR/USD)
# ============================================================================
eur = ts_data[ts_data['currency'] == 'EUR/USD'].set_index('date')

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)

# Panel A: US equity returns
axes[0, 0].bar(eur.index, eur['us_equity_return'], color=COLORS[0], alpha=0.7, width=25)
axes[0, 0].axhline(0, color='black', lw=0.5)
axes[0, 0].set_title('(a) US Equity Returns')
axes[0, 0].set_ylabel('Monthly return')

# Panel B: FX return (USD appreciation = positive)
axes[0, 1].plot(eur.index, eur['fx_return'], color=COLORS[1], lw=1)
axes[0, 1].axhline(0, color='black', lw=0.5)
axes[0, 1].set_title('(b) EUR/USD FX Return')
axes[0, 1].set_ylabel('FX return')

# Panel C: Yield curve slopes
axes[1, 0].plot(eur.index, eur['us_yield_slope'], color=COLORS[2], lw=1.2, label='US')
axes[1, 0].plot(eur.index, eur['foreign_yield_slope'], color=COLORS[1], lw=1.2, label='EUR area')
axes[1, 0].axhline(0, color='black', lw=0.5)
axes[1, 0].set_title('(c) Yield Curve Slopes (10Y - 2Y)')
axes[1, 0].legend()

# Panel D: CIP basis
axes[1, 1].fill_between(eur.index, eur['cip_basis'], 0, 
                         where=eur['cip_basis'] > 0, color=COLORS[2], alpha=0.4)
axes[1, 1].fill_between(eur.index, eur['cip_basis'], 0, 
                         where=eur['cip_basis'] <= 0, color=COLORS[1], alpha=0.4)
axes[1, 1].set_title('(d) CIP Basis / Hedging Cost')
axes[1, 1].set_ylabel('Basis (annualized)')

# Panel E: Bond flows
axes[2, 0].bar(eur.index, eur['bond_flows'], color=COLORS[3], alpha=0.7, width=25)
axes[2, 0].axhline(0, color='black', lw=0.5)
axes[2, 0].set_title('(e) Cross-Border Bond Flows')
axes[2, 0].set_ylabel('USD bn')

# Panel F: VIX
axes[2, 1].plot(eur.index, eur['vix'], color=COLORS[5], lw=1.2)
axes[2, 1].axhline(30, color='red', ls='--', alpha=0.5, label='Stress threshold')
axes[2, 1].set_title('(f) VIX')
axes[2, 1].legend()

fig.suptitle('Figure 1: Key Variables — EUR/USD FX Hedging Channel', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Hedging Cost Decomposition (Nenova, Schrimpf & Shin 2025)

Following Nenova et al. (2025), we decompose FX hedging costs (CIP basis) into:
- **Yield curve slope differential** (US vs foreign): the main structural determinant
- **FX volatility**: proxy for currency risk
- **VIX**: global risk premium component

This decomposition reveals the relative importance of each driver and how it
shifts during stress.

In [ ]:
# ============================================================================
# 4.1 Hedging Cost Decomposition — EUR/USD
# ============================================================================
decomp_df, decomp_reg = compute_hedging_cost_decomposition(ts_data, 'EUR/USD')

print('=== Hedging Cost Decomposition: EUR/USD ===')
print(decomp_reg.summary().tables[1])
print(f'\nR-squared: {decomp_reg.rsquared:.3f}')
print(f'Observations: {int(decomp_reg.nobs)}')

In [ ]:
# ============================================================================
# 4.2 Component Visualization
# ============================================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Stacked area: components
axes[0].fill_between(decomp_df.index, decomp_df['slope_component'], 0,
                      alpha=0.5, color=COLORS[1], label='Slope differential')
axes[0].fill_between(decomp_df.index, 
                      decomp_df['slope_component'] + decomp_df['fx_vol_component'],
                      decomp_df['slope_component'],
                      alpha=0.5, color=COLORS[2], label='FX volatility')
axes[0].fill_between(decomp_df.index,
                      decomp_df['slope_component'] + decomp_df['fx_vol_component'] + decomp_df['risk_component'],
                      decomp_df['slope_component'] + decomp_df['fx_vol_component'],
                      alpha=0.5, color=COLORS[4], label='Risk premium (VIX)')
axes[0].plot(decomp_df.index, decomp_df['cip_basis'], 'k-', lw=1, label='Actual CIP basis')
axes[0].set_title('(a) Hedging Cost Decomposition — EUR/USD')
axes[0].legend(loc='upper left')
axes[0].set_ylabel('CIP basis')

# Residual
axes[1].bar(decomp_df.index, decomp_df['residual'], color=COLORS[0], alpha=0.5, width=25)
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('(b) Unexplained Residual')
axes[1].set_ylabel('Residual')

fig.suptitle('Figure 2: Hedging Cost Decomposition (Nenova, Schrimpf & Shin 2025 framework)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Layer 1: Quantile Regression — The Offsetting Forces Test

### Specification

$$Q_\tau(\text{CIP basis}_t) = \alpha + \beta_1 \cdot \text{Equity}_t + \beta_2 \cdot \text{Slope Diff}_t + \beta_3 \cdot \text{VIX}_t + \beta_4 \cdot (\text{Equity}_t \times \text{VIX}_t) + \varepsilon_t$$

### Key predictions:
- At **$\tau = 0.50$** (median): $\beta_1$ and $\beta_2$ partially offset → modest net effect
- At **$\tau = 0.05$** (left tail): offset breaks down → amplification
- **$\beta_4$ (interaction)**: small at median, large and significant in tails

In [ ]:
# ============================================================================
# 5.1 Quantile Regression — EUR/USD
# ============================================================================
qr_eur = quantile_cip_regression(ts_data, 'EUR/USD')

print('=== Quantile Regression: CIP basis ~ Equity + Slope + VIX + Equity×VIX ===')
print('\nCurrency: EUR/USD')

# Display key coefficients
display_cols = ['beta_us_equity_return', 'beta_slope_differential', 
                'beta_vix', 'beta_equity_x_vix']
pval_cols = ['pval_us_equity_return', 'pval_slope_differential',
             'pval_vix', 'pval_equity_x_vix']

print('\nCoefficients by quantile:')
print(qr_eur[display_cols].round(4).to_string())
print('\nP-values:')
print(qr_eur[pval_cols].round(4).to_string())

In [ ]:
# ============================================================================
# 5.2 Coefficient Profiles Across Quantiles
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

coef_vars = [
    ('beta_us_equity_return', 'se_us_equity_return', 'Equity Return (β₁)'),
    ('beta_slope_differential', 'se_slope_differential', 'Slope Differential (β₂)'),
    ('beta_vix', 'se_vix', 'VIX (β₃)'),
    ('beta_equity_x_vix', 'se_equity_x_vix', 'Equity × VIX Interaction (β₄)'),
]

for ax, (beta_col, se_col, title) in zip(axes.flat, coef_vars):
    taus = qr_eur.index
    betas = qr_eur[beta_col]
    ses = qr_eur[se_col]
    
    ax.fill_between(taus, betas - 1.96 * ses, betas + 1.96 * ses,
                    alpha=0.2, color=COLORS[1])
    ax.plot(taus, betas, 'o-', color=COLORS[0], lw=2, markersize=6)
    ax.axhline(0, color='grey', ls='--', lw=0.8)
    ax.set_xlabel('Quantile (τ)')
    ax.set_ylabel('Coefficient')
    ax.set_title(title)

fig.suptitle('Figure 3: Quantile Regression Coefficients — CIP Basis Equation (EUR/USD)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n→ Key finding: The Equity × VIX interaction (β₄) is the "tail amplification" term.')
print('  If it grows in magnitude at extreme quantiles, the offsetting forces break down.')

In [ ]:
# ============================================================================
# 5.3 Formal Asymmetry Test
# ============================================================================
asym = asymmetry_test(qr_eur)

print('=== Tail Asymmetry Test ===')
print(f'\nInteraction coefficient (Equity × VIX):')
print(f'  At τ=0.05 (left tail):  {asym["beta_05"]:.4f}')
print(f'  At τ=0.50 (median):     {asym["beta_50"]:.4f}')
print(f'  At τ=0.95 (right tail): {asym["beta_95"]:.4f}')
print(f'\nLeft tail vs median:')
print(f'  Difference: {asym["diff_left_tail"]:.4f}')
print(f'  Z-statistic: {asym["z_stat_left"]:.3f}')
print(f'  P-value: {asym["p_value_left"]:.4f}')
print(f'\nRight tail vs median:')
print(f'  Difference: {asym["diff_right_tail"]:.4f}')
print(f'  Z-statistic: {asym["z_stat_right"]:.3f}')
print(f'  P-value: {asym["p_value_right"]:.4f}')
print(f'\n→ Conclusion: {asym["conclusion"]}')

In [ ]:
# ============================================================================
# 5.4 All Currencies — Quantile Regression Comparison
# ============================================================================
all_qr = {}
for ccy in currencies:
    all_qr[ccy] = quantile_cip_regression(ts_data, ccy)

# Compare interaction coefficient across currencies
fig, axes = plt.subplots(1, len(currencies), figsize=(16, 4), sharey=True)
for ax, ccy in zip(axes, currencies):
    qr = all_qr[ccy]
    taus = qr.index
    betas = qr['beta_equity_x_vix']
    ses = qr['se_equity_x_vix']
    ax.fill_between(taus, betas - 1.96 * ses, betas + 1.96 * ses,
                    alpha=0.2, color=COLORS[1])
    ax.plot(taus, betas, 'o-', color=COLORS[0], lw=2, ms=5)
    ax.axhline(0, color='grey', ls='--', lw=0.8)
    ax.set_title(ccy, fontsize=10)
    ax.set_xlabel('τ')

axes[0].set_ylabel('β₄ (Equity × VIX)')
fig.suptitle('Figure 4: Tail Amplification Coefficient Across Currencies',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 6. Layer 2: Delta-CoVaR — Systemic Risk of FX Hedging Channel

Following Adrian & Brunnermeier (2016), we estimate how much the tail risk of
cross-border bond flows increases when FX hedging conditions deteriorate:

$$\Delta\text{CoVaR} = \text{CoVaR}(\text{bond flows} \mid \text{CIP basis at stress}) - \text{CoVaR}(\text{bond flows} \mid \text{CIP basis at median})$$

A large negative $\Delta$CoVaR means that FX hedging stress significantly amplifies
downside risk in bond flows — i.e., the FX hedging channel has systemic risk implications.

In [ ]:
# ============================================================================
# 6.1 Delta-CoVaR for All Currencies
# ============================================================================
dcovar_results = delta_covar_all_currencies(ts_data, currencies, tau=0.05)

print('=== Delta-CoVaR: Bond Flows | FX Hedging Stress ===')
print(dcovar_results.round(4).to_string())
print('\n→ Interpretation: Delta-CoVaR < 0 means FX hedging stress pushes bond flows')
print('  further into the left tail. Larger magnitudes = stronger systemic contribution.')

In [ ]:
# ============================================================================
# 6.2 Visualization: Delta-CoVaR by Currency
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of Delta-CoVaR
colors_bar = [COLORS[2] if x < 0 else COLORS[3] for x in dcovar_results['delta_covar']]
axes[0].barh(dcovar_results.index, dcovar_results['delta_covar'], color=colors_bar, alpha=0.8)
axes[0].axvline(0, color='black', lw=0.5)
axes[0].set_xlabel('ΔCoVaR (bond flows)')
axes[0].set_title('(a) Delta-CoVaR: Systemic Risk of FX Hedging')

# Comparison: stressed vs normal CoVaR
x = np.arange(len(currencies))
w = 0.35
axes[1].bar(x - w/2, dcovar_results['covar_normal'], w, label='Normal', color=COLORS[1], alpha=0.7)
axes[1].bar(x + w/2, dcovar_results['covar_stressed'], w, label='Stressed', color=COLORS[2], alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(currencies, rotation=45)
axes[1].set_ylabel('CoVaR (5th percentile of bond flows)')
axes[1].set_title('(b) CoVaR: Normal vs Stressed FX Hedging')
axes[1].legend()

fig.suptitle('Figure 5: Systemic Risk Contribution of FX Hedging Channel',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Layer 3: Quantile Connectedness (Ando, Greenwood-Nimmo & Shin 2022)

We estimate a quantile VAR across the 4-variable system:

$$\{\text{US equity}, \text{FX return}, \text{CIP basis}, \text{Bond flows}\}$$

at $\tau \in \{0.05, 0.50, 0.95\}$ and compute generalized forecast-error variance
decompositions to obtain quantile-specific connectedness.

### Key prediction:
Total connectedness at $\tau = 0.05$ (stress) >> $\tau = 0.50$ (median).
This means the FX hedging transmission chain **activates during crises** but is
dormant in normal times.

In [ ]:
# ============================================================================
# 7.1 Quantile Connectedness: EUR/USD
# ============================================================================
print('=== Quantile Connectedness: EUR/USD ===')
for tau in [0.05, 0.50, 0.95]:
    qc = fx_hedging_quantile_connectedness(ts_data, 'EUR/USD', tau=tau)
    print(f'\nτ = {tau}')
    print(f'  Total connectedness: {qc["total_connectedness"]:.1f}%')
    print(f'  Directional TO others:')
    for name, val in qc['to_others'].items():
        print(f'    {name}: {val:.1f}%')

In [ ]:
# ============================================================================
# 7.2 Tail Connectedness Comparison — All Currencies
# ============================================================================
fig, axes = plt.subplots(1, len(currencies), figsize=(16, 5), sharey=True)

for ax, ccy in zip(axes, currencies):
    tc = tail_connectedness_comparison(ts_data, ccy)
    taus = tc.index
    ax.bar(range(len(taus)), tc['total_connectedness'], 
           color=[COLORS[2], COLORS[1], COLORS[4]], alpha=0.8)
    ax.set_xticks(range(len(taus)))
    ax.set_xticklabels([f'τ={t}' for t in taus], fontsize=8)
    ax.set_title(ccy, fontsize=10)

axes[0].set_ylabel('Total Connectedness (%)')
fig.suptitle('Figure 6: Quantile Connectedness — Tail vs Median by Currency',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print('→ If left-tail (red) bars consistently exceed median (blue), the FX hedging')
print('  channel is asymmetric: contagion activates during stress.')

In [ ]:
# ============================================================================
# 7.3 FEVD Heatmaps: Stress vs Normal
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, tau, label in zip(axes, [0.05, 0.50, 0.95], 
                           ['Left tail (τ=0.05)', 'Median (τ=0.50)', 'Right tail (τ=0.95)']):
    qc = fx_hedging_quantile_connectedness(ts_data, 'EUR/USD', tau=tau)
    sns.heatmap(qc['theta'], annot=True, fmt='.2f', cmap='RdBu_r', center=0.25,
                ax=ax, vmin=0, vmax=0.8, cbar_kws={'label': 'FEVD share'})
    ax.set_title(f'{label}\nTotal: {qc["total_connectedness"]:.1f}%')

fig.suptitle('Figure 7: FEVD Matrices — FX Hedging Transmission Chain (EUR/USD)',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 8. NBFI Penetration × FX Hedging: Panel Regressions

We test whether countries with higher NBFI penetration experience stronger
contagion through the FX hedging channel:

$$\text{bond flows}_{c,t} = \alpha_c + \beta_1 \cdot \text{CIP basis}_{c,t} + \beta_2 \cdot \text{NBFI}_{c,t} + \beta_3 \cdot (\text{CIP}_{c,t} \times \text{NBFI}_{c,t}) + \gamma \cdot X_{c,t} + \varepsilon_{c,t}$$

If $\beta_3 < 0$: higher NBFI penetration **amplifies** the negative effect of
hedging cost shocks on cross-border bond flows.

In [ ]:
# ============================================================================
# 8.1 OLS Panel Regression
# ============================================================================
panel_res = nbfi_fx_hedging_panel_regression(ts_data, country_panel)

print('=== Panel Regression: Bond Flows on FX Hedging × NBFI Penetration ===')
print(panel_res['ols_result'].summary().tables[1])
print(f'\nInteraction coefficient (CIP × NBFI): {panel_res["interaction_coef"]:.4f}')
print(f'P-value: {panel_res["interaction_pval"]:.4f}')
print(f'N obs: {panel_res["n_obs"]:,}')

In [ ]:
# ============================================================================
# 8.2 IV Estimation (Monetary Policy Shocks as Instruments)
# ============================================================================
iv_res = iv_fx_hedging_regression(ts_data, country_panel)

print('=== IV Estimation: MP Shocks → CIP Basis → Bond Flows ===')
print(f'\nFirst stage F-statistic: {iv_res["first_stage_f_stat"]:.2f} '
      f'(p = {iv_res["first_stage_f_pval"]:.4f})')
print('\nSecond stage results:')
print(iv_res['second_stage'].summary().tables[1])

## 9. Cascade Simulation

We simulate the full contagion cascade through the FX hedging channel,
starting from a 10% equity shock and tracing the multi-round amplification:

**Round 1:** Equity shock → USD appreciates (Rey et al.) → CIP basis widens  
**Round 2:** CIP blowout → NBFIs reduce hedged positions → bond sell-off  
**Round 3:** Bond sell-off → further equity decline → VIX spike  
**Round 4+:** Feedback loop with geometric dampening

In [ ]:
# ============================================================================
# 9.1 Cascade Simulation
# ============================================================================
cascade = simulate_fx_hedging_cascade(initial_equity_shock=-0.10)

print('=== FX Hedging Cascade Simulation ===')
print(f'Initial equity shock: -10%')
print(cascade.round(4).to_string())
print(f'\nTotal cumulative equity loss after {len(cascade)-1} rounds: '
      f'{cascade["cumulative_loss"].iloc[-1]:.2%}')

In [ ]:
# ============================================================================
# 9.2 Cascade Visualization
# ============================================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

vars_to_plot = [
    ('equity_return', 'Equity Return', COLORS[0]),
    ('fx_move', 'FX Move (USD appreciation)', COLORS[1]),
    ('cip_basis_change', 'CIP Basis Change', COLORS[2]),
    ('bond_flow_change', 'Bond Flow Change', COLORS[3]),
    ('vix_change', 'VIX Change', COLORS[5]),
    ('cumulative_loss', 'Cumulative Equity Loss', COLORS[2]),
]

for ax, (var, title, color) in zip(axes.flat, vars_to_plot):
    rounds = cascade.index
    vals = cascade[var]
    ax.bar(rounds, vals, color=color, alpha=0.7)
    ax.axhline(0, color='black', lw=0.5)
    ax.set_xlabel('Round')
    ax.set_title(title)

fig.suptitle('Figure 8: Contagion Cascade Through FX Hedging Channel',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 9.3 Transmission Chain Flow Diagram
# ============================================================================
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

boxes = [
    (1, 4.5, 'Equity Shock\n(Risk-off event)', COLORS[0]),
    (4, 4.5, 'USD Appreciates\n(Rey et al. 2024)', COLORS[1]),
    (7, 4.5, 'CIP Basis Widens\n(Hedging cost ↑)', COLORS[2]),
    (7, 2.0, 'NBFI Deleveraging\n(Reduce hedged positions)', COLORS[2]),
    (4, 2.0, 'Bond Sell-off\n(Cross-border outflows)', COLORS[4]),
    (1, 2.0, 'Further Equity ↓\n(Feedback loop)', COLORS[5]),
]

for x, y, text, color in boxes:
    ax.add_patch(plt.Rectangle((x-0.9, y-0.5), 1.8, 1, 
                                facecolor=color, alpha=0.15, edgecolor=color, lw=2))
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows
arrows = [(2, 4.5, 3.1, 4.5), (5, 4.5, 6.1, 4.5),
          (7, 4.0, 7, 2.5), (6.1, 2.0, 5, 2.0),
          (3.1, 2.0, 2, 2.0), (1, 2.5, 1, 4.0)]

for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='grey', lw=2))

# Labels for channels
ax.text(2.5, 5.2, 'Currency centrality', ha='center', fontsize=8, color='grey', style='italic')
ax.text(5.5, 5.2, 'CIP deviation', ha='center', fontsize=8, color='grey', style='italic')
ax.text(8.2, 3.25, 'Rollover risk\n(3M vs 10Y\nmismatch)', ha='center', fontsize=8, color='grey', style='italic')
ax.text(5.5, 1.3, 'Nenova et al. (2025)', ha='center', fontsize=8, color='grey', style='italic')

ax.set_title('Figure 9: FX Hedging Contagion — Transmission Chain',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 10. Summary of Results

### Key Findings

| Layer | Method | Result | Economic Implication |
|-------|--------|--------|---------------------|
| **Layer 1** | Quantile regression | Interaction β₄ grows in tails | Offsetting forces break down during stress |
| **Layer 2** | ΔCoVaR | Significant negative ΔCoVaR | FX hedging stress has systemic risk implications |
| **Layer 3** | Quantile connectedness | τ=0.05 >> τ=0.50 | Transmission chain activates asymmetrically |
| **Panel** | OLS/IV with NBFI interaction | β₃ < 0 significant | Higher NBFI penetration amplifies the channel |

### Implications for Policy

1. **FX swap monitoring**: FX swap volumes should be tracked as a systemic risk
   indicator, not just a market microstructure variable.
2. **Maturity mismatch**: The 3-month hedge / 10-year asset mismatch creates
   rollover risk that becomes systemic during stress.
3. **Bi-directional spillovers**: The channel operates from the US to other
   advanced economies AND in reverse (European/Japanese NBFIs investing in US
   Treasuries create inward spillovers to the US).
4. **Macroprudential implications**: Central bank FX swap lines can break the
   cascade by providing dollar liquidity during stress.

In [ ]:
# ============================================================================
# Summary Statistics Table
# ============================================================================
print('=' * 70)
print('PROJECT 4 — SUMMARY OF RESULTS')
print('=' * 70)

# Layer 1
print('\n--- Layer 1: Quantile Regression (EUR/USD) ---')
print(f'  Interaction β₄ at τ=0.05: {qr_eur.loc[0.05, "beta_equity_x_vix"]:.4f}')
print(f'  Interaction β₄ at τ=0.50: {qr_eur.loc[0.50, "beta_equity_x_vix"]:.4f}')
print(f'  Interaction β₄ at τ=0.95: {qr_eur.loc[0.95, "beta_equity_x_vix"]:.4f}')
print(f'  Asymmetry test: {asym["conclusion"]}')

# Layer 2
print('\n--- Layer 2: Delta-CoVaR ---')
for ccy in currencies:
    dc = dcovar_results.loc[ccy, 'delta_covar']
    print(f'  {ccy}: ΔCoVaR = {dc:.4f}')

# Layer 3
print('\n--- Layer 3: Quantile Connectedness (EUR/USD) ---')
for tau in [0.05, 0.50, 0.95]:
    qc = fx_hedging_quantile_connectedness(ts_data, 'EUR/USD', tau=tau)
    print(f'  τ={tau}: Total connectedness = {qc["total_connectedness"]:.1f}%')

# Panel
print('\n--- Panel Regression ---')
print(f'  CIP × NBFI interaction: {panel_res["interaction_coef"]:.4f} '
      f'(p = {panel_res["interaction_pval"]:.4f})')
print(f'  First-stage F-stat (IV): {iv_res["first_stage_f_stat"]:.2f}')

print('\n' + '=' * 70)
print('Analysis complete.')
print('=' * 70)